# NAIC WP7 UC6: Multi-Modal Optimization with SHGA**Scaling Hybrid Genetic Algorithm for finding all optima of multimodal functions**---This notebook demonstrates the SHGA (Scaling Hybrid Genetic Algorithm) on theCEC2013 niching benchmark suite. It is fully self-contained and runs withoutexternal dependencies beyond NumPy, SciPy, deap, and the included `mmo` package.**Key concepts:**- Multimodal optimization: finding *all* optima, not just one- Deterministic crowding for diversity preservation- CMA-ES local refinement for precision- Population scaling for progressive discovery**Reference:** Johannsen *et al.* (2022)

In [ ]:
# ============================================================================# Environment Setup# ============================================================================import matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltimport numpy as npimport pandas as pdimport warningsimport sysimport osimport timeimport multiprocessingwarnings.filterwarnings("ignore", message="A class named", category=RuntimeWarning)warnings.filterwarnings("ignore", message="invalid value", category=RuntimeWarning)np.seterr(invalid="ignore")# Add CEC2013 benchmark to pathsys.path.append(os.path.abspath("benchmarks/CEC2013/python3"))print(f"Python: {sys.version}")print(f"NumPy: {np.__version__}")print(f"CPU cores: {multiprocessing.cpu_count()}")print(f"Working directory: {os.getcwd()}")

---## Part 1: ConfigurationThese parameters control the benchmark run. Adjust as needed:

In [ ]:
# ============================================================================# User-Configurable Parameters# ============================================================================RUN_FUNCTIONS = list(range(1, 8))  # CEC2013 functions F1–F7 (1D and 2D)MAX_RUN = 5                         # Number of SHGA iterations per functionACCURACY = 0.0001                   # Tolerance for counting global optima

---## Part 2: The CEC2013 Benchmark SuiteThe CEC2013 niching benchmark has 20 functions with dimensions 1–20.Each function has a known number of global optima and a function evaluation budget.| Function | Name | Dim | Global Optima ||----------|------|-----|---------------|| F1 | Five-Uneven-Peak Trap | 1 | 2 || F2 | Equal Maxima | 1 | 5 || F3 | Uneven Decreasing Maxima | 1 | 1 || F4 | Himmelblau | 2 | 4 || F5 | Six-Hump Camel Back | 2 | 2 || F6 | Shubert | 2 | 18 || F7 | Vincent | 2 | 36 |Performance is measured by **Peak Ratio (NR)**: the fraction of known global optima found.

In [ ]:
# ============================================================================# Import SHGA Components# ============================================================================from cec2013.cec2013 import CEC2013, how_many_goptimafrom mmo.domain import Domainfrom mmo.minimize import MultiModalMinimizerfrom mmo import function, configimport mmo.integrate as integratefrom deap import creator# Function metadata for displayFUNCTIONS_META = {    1: "Five-Uneven-Peak", 2: "Equal Maxima", 3: "Uneven Decreasing Max.",    4: "Himmelblau", 5: "Six-Hump Camel Back", 6: "Shubert", 7: "Vincent",    8: "Shubert (3D)", 9: "Vincent (3D)", 10: "Modified Rastrigin",}print("SHGA components loaded successfully.")

---## Part 3: Verify CEC2013 FunctionsQuick sanity check — evaluate each function at a test point and show its properties:

In [ ]:
# ============================================================================# Verify CEC2013 Functions# ============================================================================print(f"{'F#':<4} {'Name':<25} {'Dim':<5} {'Optima':<8} {'Budget':<10} {'f(ones)':<12}")print("=" * 70)for i in RUN_FUNCTIONS:    f = CEC2013(i)    dim = f.get_dimension()    info = f.get_info()    x_test = np.ones(dim)    val = f.evaluate(x_test)    name = FUNCTIONS_META.get(i, f"F{i}")    print(f"F{i:<3} {name:<25} {dim:<5} {info['nogoptima']:<8} {info['maxfes']:<10} {val:<12.4f}")

---## Part 4: SHGA Algorithm ExplanationThe SHGA operates in two phases within an outer loop:1. **Global Niching GA (Exploration):**   Uses deterministic crowding — offspring compete only with their nearest parent,   preserving diversity across multiple basins of attraction.2. **Local CMA-ES Refinement (Exploitation):**   After clustering, each seed point is refined using CMA-ES, which adapts its   search distribution for precise convergence.3. **Population Scaling:**   The population grows incrementally across iterations, discovering additional   optima as search progresses.---## Part 5: Run the Benchmark

In [ ]:
# ============================================================================# Run SHGA for Each Benchmark Function# ============================================================================results_list = []for i in RUN_FUNCTIONS:    t_start = time.time()    f = CEC2013(i)    info = f.get_info()    dim = f.get_dimension()    BUDGET = info["maxfes"]    lb = np.array([f.get_lbound(k) for k in range(dim)])    ub = np.array([f.get_ubound(k) for k in range(dim)])    dom = Domain(boundary=[lb, ub])    num_of_sols = []    num_of_global_found = []    num_of_fev = []    final_iteration = None    for iteration in MultiModalMinimizer(            f=f, domain=dom, verbose=0, budget=BUDGET, max_iter=MAX_RUN):        final_iteration = iteration        num_of_sols.append(iteration.n_sol)        num_of_fev.append(iteration.n_fev)        X = iteration.x        count, seeds = how_many_goptima(X, f, ACCURACY)        num_of_global_found.append(count)    elapsed = time.time() - t_start    num_global = f.get_no_goptima()    avg_global = np.mean(num_of_global_found) if num_of_global_found else 0    NR = avg_global / num_global if num_global > 0 else 0    name = FUNCTIONS_META.get(i, f"F{i}")    print(f"F{i} {name}: NR={NR:.3f} ({avg_global:.1f}/{num_global} optima), {elapsed:.1f}s")    results_list.append({        "Function": f"F{i}",        "Name": name,        "Dim": dim,        "Known Optima": num_global,        "Found (avg)": f"{avg_global:.1f}",        "Peak Ratio (NR)": f"{NR:.3f}",        "Avg FEV": f"{np.mean(num_of_fev):.0f}" if num_of_fev else "0",        "Time (s)": f"{elapsed:.1f}",    })print("\nBenchmark complete.")

---## Part 6: Results Summary

In [ ]:
# ============================================================================# Results Table# ============================================================================df = pd.DataFrame(results_list)print("=" * 80)print("SHGA BENCHMARK RESULTS")print(f"Parameters: MAX_RUN={MAX_RUN}, ACCURACY={ACCURACY}")print("=" * 80)print(df.to_string(index=False))print("=" * 80)

In [ ]:
# ============================================================================# Peak Ratio Bar Chart# ============================================================================fig, axes = plt.subplots(1, 2, figsize=(14, 5))# Bar chart: Peak Ratio per functionnr_values = [float(r["Peak Ratio (NR)"]) for r in results_list]func_labels = [r["Function"] for r in results_list]colors = ['#2ecc71' if nr >= 0.8 else '#f39c12' if nr >= 0.5 else '#e74c3c' for nr in nr_values]axes[0].bar(func_labels, nr_values, color=colors, edgecolor='black', linewidth=0.5)axes[0].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Perfect (NR=1.0)')axes[0].set_ylabel('Peak Ratio (NR)')axes[0].set_title('SHGA Peak Ratio by CEC2013 Function')axes[0].set_ylim(0, 1.15)axes[0].legend()# Bar chart: Timing per functiontimes = [float(r["Time (s)"]) for r in results_list]axes[1].bar(func_labels, times, color='#3498db', edgecolor='black', linewidth=0.5)axes[1].set_ylabel('Time (seconds)')axes[1].set_title('Processing Time per Function')plt.tight_layout()plt.savefig('results/benchmark_results.png', dpi=150, bbox_inches='tight')plt.show()print("Saved: results/benchmark_results.png")

---## Part 7: Interpretation- **Peak Ratio (NR) = 1.0** means all known global optima were found- **Green bars** (NR >= 0.8): strong performance- **Orange bars** (NR 0.5–0.8): partial success, may need more iterations- **Red bars** (NR < 0.5): function is challenging for current settingsLower-dimensional functions (F1–F5) typically achieve NR close to 1.0.Higher-dimensional functions (F6+) with many optima are progressively harder.---## Part 8: Summary### Key Takeaways1. **SHGA finds multiple optima** across diverse benchmark landscapes2. **Deterministic crowding** preserves population diversity without explicit niche radius3. **CMA-ES refinement** provides high-precision convergence to each optimum4. **Population scaling** allows progressive discovery of additional peaks### Extending This Work- Increase `MAX_RUN` for more thorough search- Extend `RUN_FUNCTIONS` to F8–F20 for higher-dimensional challenges- Use `MultiModalMinimizerParallel` for multi-core acceleration- Apply SHGA to custom objective functions via the `Domain` + function interface

In [ ]:
# ============================================================================# Final Summary# ============================================================================import globprint("Generated output files:")for f in sorted(glob.glob('results/*.png') + glob.glob('results/*.csv')):    print(f"  {f}")total_nr = np.mean([float(r["Peak Ratio (NR)"]) for r in results_list])total_time = sum([float(r["Time (s)"]) for r in results_list])print(f"\nOverall average Peak Ratio: {total_nr:.3f}")print(f"Total benchmark time: {total_time:.1f}s")print(f"Functions evaluated: {len(RUN_FUNCTIONS)}")print("\nNotebook execution complete.")